In [2]:
import numpy as np

In [92]:
words = ["saya", "makan", "nasi", "pake", "ayam"]
word_size = len(words)

target = np.eye(word_size)
def label(words: list):
    return {idx:value for idx, value in enumerate(words)}

label(words)

def init_embedding(vocab_size, embed_dim):
    # Pakai Xavier/Glorot initialization supaya nilainya gak kegedean
    return np.random.randn(vocab_size, embed_dim) * np.sqrt(1.0 / vocab_size)

# Contoh penggunaan:
embed_dim = 4  # Tiap kata sekarang punya 8 dimensi 'ciri khas'

word_embedings = init_embedding(word_size, embed_dim)

def embedding_forward(word_idx, E):
    """
    word_idx: integer index dari kata (misal: 'saya' = 0)
    E: Matriks embedding
    """
    # Secara teknis ini sama dengan: vocabulary[word_idx] @ E
    # Tapi jauh lebih cepat kalau kita ambil langsung barisnya
    return E[word_idx, :]


In [93]:
# --- Konfigurasi ---
h_size = embed_dim
c_size = embed_dim
fc_hidden_size = 10 # Lu sebut '10' di kode lu tadi

# --- Inisialisasi State ---
h_init = np.zeros(h_size)
c_init = np.zeros(c_size)

# --- 1. Recurrent Weights (LSTM Gates) ---
# Output gate harus seukuran c_size karena akan di-update ke cell state
# Inputnya adalah gabungan h_past + x_embed
concat_size = h_size + embed_dim
scale = np.sqrt(1.0 / concat_size)

wf = np.random.randn(c_size, concat_size) * scale
wi = np.random.randn(c_size, concat_size) * scale
wo = np.random.randn(c_size, concat_size) * scale
wc = np.random.randn(c_size, concat_size) * scale

# --- 2. Fully Connected Weights ---
# Layer 1: Dari Hidden State (h_size) ke FC Hidden (10)
w1 = np.random.randn(fc_hidden_size, h_size) * np.sqrt(2.0 / h_size)
b1 = np.zeros(fc_hidden_size)

# Layer 2: Dari FC Hidden (10) ke Output FC (c_size?) 
# Note: Di kode lu, w2 ngeluarin dimensi 2. Gue asumsikan ini balik ke h_size/c_size.
w2 = np.random.randn(h_size, fc_hidden_size) * np.sqrt(2.0 / fc_hidden_size)
b2 = np.zeros(h_size)

# --- 3. Prediction Weight (Softmax Layer) ---
# Dari output FC terakhir ke jumlah kosakata (word_size)
why = np.random.randn(word_size, h_size) * np.sqrt(1.0 / h_size)
by = np.zeros(word_size)

#Optimizer Params
mwf = np.zeros_like(wf)
vwf = np.zeros_like(wf)

mwi = np.zeros_like(wi)
vwi = np.zeros_like(wi)

mwo = np.zeros_like(wo)
vwo = np.zeros_like(wo)

mwc = np.zeros_like(wc)
vwc = np.zeros_like(wc)

mw1 = np.zeros_like(w1)
vw1 = np.zeros_like(w1)

mw2 = np.zeros_like(w2)
vw2 = np.zeros_like(w2)

mb2 = np.zeros_like(b2)
vb2 = np.zeros_like(b2)

mb1 = np.zeros_like(b1)
vb1 = np.zeros_like(b1)

mwhy = np.zeros_like(why)
vwhy = np.zeros_like(why)

mby = np.zeros_like(by)
vby = np.zeros_like(by)

beta_1 = 0.9
beta_2 = 0.999
e = 10**-8
lr = 0.001

In [ ]:

def tanh_deriv(x: np.ndarray, from_activation: bool = False) -> np.ndarray:
    x = np.asarray(x)
    if from_activation:
        return 1.0 - x * x
    t = np.tanh(x)
    return 1.0 - t * t

def sigmoid(x: np.ndarray) -> np.ndarray:
    """Sigmoid activation."""
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_deriv(x: np.ndarray) -> np.ndarray:
    """Derivative of sigmoid: s * (1 - s) where s = sigmoid(x)."""
    x = np.asarray(x)
    s = sigmoid(x)
    return s * (1.0 - s)

def relu(x: np.ndarray) -> np.ndarray:
    """ReLU activation."""
    x = np.asarray(x)
    return np.maximum(0, x)

def relu_deriv(x: np.ndarray) -> np.ndarray:
    """Derivative of ReLU: 1 for x>0, 0 otherwise (including x==0)."""
    x = np.asarray(x)
    return (x > 0).astype(float)

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = np.asarray(x)
    x_max = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - x_max)
    return e / np.sum(e, axis=axis, keepdims=True)

def forward(h_past: np.ndarray, x: np.ndarray, c_past: np.ndarray):
    x = np.concatenate((h_past, x), axis=0)

    ft = wf @ x
    aft = sigmoid(ft)

    it = wi @ x
    ait = sigmoid(it)

    ot = wo @ x
    aot = sigmoid(ot)

    cell_state = wc @ x
    acell_state = np.tanh(cell_state)


    ct = (aft * c_past) + (ait * acell_state)
    act = np.tanh(ct)
    ht = aot * act

    #Fully Connected
    z1 = w1 @ ht + b1
    az1 = relu(z1)

    z2 = w2 @ az1 + b2
    az2 = relu(z2)

    #Prediction
    yt = softmax(why @ az2 + by)

    return {"yt": yt, "ht": ht, "ct": ct}, {"ft": ft, "it": it, "ot": ot, "cell_state": cell_state, "z1": z1, "z2": z2}, {"aft": aft, "ait": ait, "aot": aot, "acell_state": acell_state, "act": act, "az1": az1, "az2": az2}, {"x": x}

def forward_predict(x: np.ndarray):
    x = np.concatenate((h_init, x), axis=0)

    ft = wf @ x
    aft = sigmoid(ft)

    it = wi @ x
    ait = sigmoid(it)

    ot = wo @ x
    aot = sigmoid(ot)

    cell_state = wc @ x
    acell_state = np.tanh(cell_state)


    ct = (aft * c_init) + (ait * acell_state)
    act = np.tanh(ct)
    ht = aot * act

    #Fully Connected
    z1 = w1 @ ht + b1
    az1 = relu(z1)

    z2 = w2 @ az1 + b2
    az2 = relu(z2)

    #Prediction
    yt = softmax(why @ az2 + by)

    return yt

def backward(out_params_t: dict, support_params_t: dict,activation_params_t: dict, target: np.ndarray, xt: dict):
    total_dwf = np.zeros_like(wf)
    total_dwi = np.zeros_like(wi)
    total_dwo = np.zeros_like(wo)
    total_dwc = np.zeros_like(wc)

    total_dw1 = np.zeros_like(w1)
    total_dw2 = np.zeros_like(w2)

    total_db1 = np.zeros_like(b1)
    total_db2 = np.zeros_like(b2)

    total_dwhy = np.zeros_like(why)
    total_dby = np.zeros_like(by)

    total_dht = np.zeros_like(h_init)
    total_dct = np.zeros_like(c_init)
    for i in range(len(out_params_t["yt"]), 0, -1):
        dy = out_params_t["yt"][i] - target[i]
        dwhy = np.outer(dy, activation_params_t["az2"][i])
        dby = dy * 1

        daz2 = dy @ why 
        dz2 = daz2 * relu_deriv(support_params_t["z2"][i])

        dw2 = np.outer(dz2, activation_params_t["az1"][i])
        db2 = dz2 * 1

        daz1 = dz2 @ w2
        dz1 = daz1 * relu_deriv(support_params_t["z1"][i])

        dw1 = np.outer(dz1, out_params_t["ht"][i])
        db1 = dz1 * 1       
        dht = dz1 @ w1 + total_dht 

        daot = dht * activation_params_t["act"][i]
        dot = daot * sigmoid_deriv(support_params_t["ot"][i])
        dwo = np.outer(dot, xt[i])
        dxo = dot @ wo 
        dh_prev_o = dxo[:h_size]

        dact = dht * activation_params_t["aot"][i]
        dct = dact * tanh_deriv(out_params_t["ct"][i]) + total_dct

        daft = dct * out_params_t["ct"][i-1]
        dft = daft * sigmoid_deriv(support_params_t["ft"][i])
        dwf = np.outer(dft, xt[i])
        dxf = dft @ wf
        dh_prev_f = dxf[:h_size]

        dc_past = dct * activation_params_t["aft"][i]
        
        dait = dct * activation_params_t["acell_state"][i]
        dit = dait * sigmoid_deriv(support_params_t["it"][i])
        dwi = np.outer(dit, xt[i])
        dxi = dit @ wi
        dh_prev_i = dxi[:h_size]
        
        dacell_state = dct * activation_params_t["ait"][i]
        dcell_state = dacell_state * tanh_deriv(support_params_t["cell_state"][i])
        dwc = np.outer(dcell_state, xt[i])
        dxcell_state = dcell_state @ wc
        dh_prev_cell_state = dxcell_state[:h_size]
        
        total_dwf += dwf
        total_dwi += dwi
        total_dwo += dwo
        total_dwc += dwc
        total_dw1 += dw1
        total_dw2 += dw2
        total_db1 += db1
        total_db2 += db2
        total_dwhy += dwhy
        total_dby += dby

        dh = dh_prev_o + dh_prev_f + dh_prev_i + dh_prev_cell_state
        total_dht += dh

        total_dct += dc_past
    
    return {"dwf": total_dwf, "dwi": total_dwi, "dwo": total_dwo, "dwc": total_dwc, "dw1": total_dw1, "dw2": total_dw2, "db1": total_db1, "db2": total_db2, "dwhy": total_dwhy, "dby": total_dby}

def categorical_cross_entropy(y_pred: np.ndarray,
                              y_true: np.ndarray,
                              eps: float = 1e-12) -> float:
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    loss = -np.sum(y_true * np.log(y_pred))
    return float(loss)

def adam_optimizer(
    m, v, theta_prev, dtheta, t
):  # Tambah parameter theta_prev
    # 1. Hitung Moment (M dan V yang baru)
    m_new = (beta_1 * m) + ((1 - beta_1) * dtheta)
    v_new = (beta_2 * v) + ((1 - beta_2) * np.square(dtheta))
    # 2. Koreksi Bias
    m_hat = m_new / (1 - (beta_1**t))
    v_hat = v_new / (1 - (beta_2**t))
    # 3. Hitung Pembaruan (Update Step)
    update = (lr / (np.sqrt(v_hat) + e)) * m_hat
    # 4. Terapkan Pembaruan
    theta_new = theta_prev - update
    # Kembalikan M dan V yang BARU, serta Theta yang BARU
    return m_new, v_new, theta_new


In [95]:
import time
epochs = 1000
t = 0

for i in range(epochs):
    t+=1
    out_params_t, support_params_t, activation_params_t = {"ht":{}, "ct":{}, "yt":{}}, {"ft":{}, "it":{}, "ot":{}, "cell_state":{}, "z1":{}, "z2":{}}, {"aft":{}, "ait":{}, "aot":{}, "acell_state":{},"act": {}, "az1":{}, "az2":{}}
    xt = {}
    out_params_t["ht"][0] = h_init
    out_params_t["ct"][0] = c_init

    avg_loss = []
    for idx in range(len(target)-1):
        out_params, support_params, activation_params, x = forward(out_params_t["ht"][idx], word_embedings[idx], out_params_t["ct"][idx])
        out_params_t["yt"][idx+1] = out_params["yt"]
        out_params_t["ht"][idx+1] = out_params["ht"]
        out_params_t["ct"][idx+1] = out_params["ct"]

        support_params_t["ft"][idx+1] = support_params["ft"]
        support_params_t["it"][idx+1] = support_params["it"]
        support_params_t["ot"][idx+1] = support_params["ot"]
        support_params_t["cell_state"][idx+1] = support_params["cell_state"]
        support_params_t["z1"][idx+1] = support_params["z1"]
        support_params_t["z2"][idx+1] = support_params["z2"]

        activation_params_t["aft"][idx+1] = activation_params["aft"]
        activation_params_t["ait"][idx+1] = activation_params["ait"]
        activation_params_t["aot"][idx+1] = activation_params["aot"]
        activation_params_t["acell_state"][idx+1] = activation_params["acell_state"]
        activation_params_t["act"][idx+1] = activation_params["act"]
        activation_params_t["az1"][idx+1] = activation_params["az1"]
        activation_params_t["az2"][idx+1] = activation_params["az2"]

        xt[idx+1] = x["x"]

        loss = avg_loss.append(categorical_cross_entropy(out_params["yt"], target[idx+1]))

    print(f"Loss: {sum(avg_loss) / len(avg_loss)}")
    d = backward(out_params_t, support_params_t, activation_params_t, target, xt)

    mwhy, vwhy, why = adam_optimizer(mwhy, vwhy, why, d["dwhy"], t)
    mby, vby, by = adam_optimizer(mby, vby, by, d["dby"], t)
    mw2, vw2, w2 = adam_optimizer(mw2, vw2, w2, d["dw2"], t)
    mw1, vw1, w1 = adam_optimizer(mw1, vw1, w1, d["dw1"], t)
    mb2, vb2, b2 = adam_optimizer(mb2, vb2, b2, d["db2"], t)
    mb1, vb1, b1 = adam_optimizer(mb1, vb1, b1, d["db1"], t)
    mwf, vwf, wf = adam_optimizer(mwf, vwf, wf, d["dwf"], t)
    mwi, vwi, wi = adam_optimizer(mwi, vwi, wi, d["dwi"], t)
    mwo, vwo, wo = adam_optimizer(mwo, vwo, wo, d["dwo"], t)
    mwc, vwc, wc = adam_optimizer(mwc, vwc, wc, d["dwc"], t)


print(forward_predict(word_embedings[2]))

Loss: 1.6173422869705774
Loss: 1.6154945094121183
Loss: 1.6143751164786675
Loss: 1.6133527271300279
Loss: 1.612344119049933
Loss: 1.6113477447349858
Loss: 1.6103624830842753
Loss: 1.6094589298267725
Loss: 1.608650874296893
Loss: 1.6078468539790176
Loss: 1.6070478238566253
Loss: 1.606254309867454
Loss: 1.6054666090159424
Loss: 1.6046848808803937
Loss: 1.6039091908391332
Loss: 1.6031395315620744
Loss: 1.6023758361309473
Loss: 1.601617987895124
Loss: 1.600865828614607
Loss: 1.600119165265102
Loss: 1.599377775494258
Loss: 1.5987200749143842
Loss: 1.5981845694703223
Loss: 1.5976380200099365
Loss: 1.5970833867756662
Loss: 1.596522772187398
Loss: 1.5959577732412125
Loss: 1.5953896479857494
Loss: 1.5948194073925104
Loss: 1.5942478719418243
Loss: 1.5936757094440608
Loss: 1.5931034620003017
Loss: 1.5925315662794297
Loss: 1.5919603694935467
Loss: 1.5913901425053196
Loss: 1.5908210909544638
Loss: 1.590253364949287
Loss: 1.5896851934371432
Loss: 1.589116395701999
Loss: 1.5885487088695158
Loss: 1.58